# Exploratory Data Analysis (EDA)
## Author: Yangshunshun (Luna) Lu


## Introduction

**1. What was done:**
This notebook integrates micro-level music characteristics (Spotify audio features like `valence`, `energy` and NLP lyrics features like `text_sentiment`) with macro-level societal data (weekly COVID-19 Google Search trends across the US and Europe). The EDA pipeline encompasses univariate distribution checks, time-series alignment, bivariate correlation mapping, exploratory statistical testing (Welch's T-tests and simple OLS regressions), and final assumption validation (VIF checks).

**2. Why it was done:**
The primary objective is to visually and statistically explore the relationship between global societal crises (the pandemic) and pop culture consumption (music features). Furthermore, this EDA acts as a rigorous statistical prerequisite—checking for linearity and multicollinearity—to ensure the data is robust enough for the final Multiple Linear Regression (MLR) model (`trend ~ sentiment + melody features`).

**3. Key Findings & Results:**
* **The "Escapism" Phenomenon**: While binary T-tests (High vs. Low COVID periods) yielded no significant differences, continuous Simple OLS regressions revealed a highly significant **positive** relationship. Counterintuitively, as COVID-19 search trends spiked, the charting music consumed became significantly *happier* and more *positive* in both lyrics ($p < 0.01$) and melody ($p < 0.001$).
* **Model Readiness**: The Variance Inflation Factor (VIF) analysis confirmed the absence of severe multicollinearity among the chosen predictors. The highest VIF score observed was `energy` at 3.107 (well below the problematic threshold of 5), indicating that all 7 audio and text features should be incorporated into the final multiple regression model.

**4. AI Disclosure:**
Gemini used in debugging.

In [ ]:
#! pip install statsmodels
#! pip install openpyxl


   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   ---------------------------------------- 2/2 [openpyxl]



In [ ]:
# ==========================================
# Phase 1: Setup & Imports
# ==========================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import statsmodels.api as sm
import statsmodels.formula.api as smf
import statsmodels.stats.api as sms
from scipy import stats

# Global plotting settings
sns.set_theme(style="whitegrid", palette="muted")
out_dir = os.path.join('..', 'result', 'EDA')
os.makedirs(out_dir, exist_ok=True)

print("✅ Libraries imported and output directory created.")

✅ Libraries imported and output directory created.


In [ ]:
# ==========================================
# Phase 2: Data Loading, Time Transformation & Merging
# ==========================================
print("1. Loading datasets...")
DATA_DIR = os.path.join('..', '..', 'Data', 'cleaned_data')
music_file = os.path.join(DATA_DIR, 'lyricsFeatures_mergeERUS_clean_enonly_token_pos_ldaTfidf.csv')

df_music = pd.read_csv(music_file)

# Load Trends Data
df_trend_eu = pd.read_csv('../../Data/Google Trends/covid_trends_europe.csv')
df_trend_us = pd.read_csv('../../Data/Google Trends/google_trends_us_weekly.csv')

print("2. Formatting Trends Data...")
# Standardize US trends to match Europe format
df_trend_us['country'] = 'USA'
df_trend_us = df_trend_us[['date', 'COVID', 'country']]

# Combine US and Europe trends
df_trend = pd.concat([df_trend_eu, df_trend_us], ignore_index=True)

print("3. Time Transformation (Aligning by Year-Week)...")
# Convert all dates to datetime
df_music['date'] = pd.to_datetime(df_music['date'], errors='coerce')
df_trend['date'] = pd.to_datetime(df_trend['date'], errors='coerce')

# Drop invalid dates
df_music = df_music.dropna(subset=['date'])
df_trend = df_trend.dropna(subset=['date'])

# Extract Year-Week for merging (e.g., "2020-W12")
df_music['year_week'] = df_music['date'].dt.strftime('%Y-W%V')
df_trend['year_week'] = df_trend['date'].dt.strftime('%Y-W%V')

# Since trends might be weekly, let's group trends by week and country to avoid duplicates
df_trend_weekly = df_trend.groupby(['year_week', 'country'])['COVID'].mean().reset_index()

print("4. Merging Music Features with Macro Trends...")
# Merge datasets on 'year_week' and 'country'
# We use inner join to keep only tracks that have matching trend data
df_merged = pd.merge(df_music, df_trend_weekly, on=['year_week', 'country'], how='inner')
df_merged.rename(columns={'COVID': 'covid_trend'}, inplace=True)

print(f"✅ Merge successful! Final dataset shape: {df_merged.shape}")

1. Loading datasets...


2. Formatting Trends Data...
3. Time Transformation (Aligning by Year-Week)...
4. Merging Music Features with Macro Trends...
✅ Merge successful! Final dataset shape: (13527, 45)


In [6]:
# ==========================================
# Phase 3: Stage 1 EDA - Distributions & Time Series
# ==========================================
print("Generating Stage 1 Visualizations...")

# 1. Distribution of COVID Trend
plt.figure(figsize=(10, 5))
sns.histplot(df_merged['covid_trend'], bins=50, kde=True, color='coral')
plt.title('Distribution of COVID Search Trends', fontsize=16, fontweight='bold')
plt.xlabel('COVID Trend Index')
plt.ylabel('Frequency (Tracks)')
plt.tight_layout()
plt.savefig(os.path.join(out_dir, '7A_COVID_Trend_Distribution.png'), dpi=300)
plt.close()

# 2. Time-Series: COVID Trend vs Average Sentiment
# Group by time for the plot
time_agg = df_merged.groupby('year_week').agg({
    'covid_trend': 'mean',
    'text_sentiment': 'mean',
    'valence': 'mean'
}).reset_index()

# Convert year_week back to a proxy datetime for plotting
time_agg['plot_date'] = pd.to_datetime(time_agg['year_week'].astype(str) + '-0', format='%Y-W%W-%w', errors='coerce')
time_agg = time_agg.dropna(subset=['plot_date']).sort_values('plot_date')

fig, ax1 = plt.subplots(figsize=(14, 6))

# Plot COVID trend on left Y-axis
color = 'tab:red'
ax1.set_xlabel('Time (Year-Week)', fontsize=12)
ax1.set_ylabel('COVID Search Trend', color=color, fontsize=12)
ax1.plot(time_agg['plot_date'], time_agg['covid_trend'], color=color, linewidth=2.5, label='COVID Trend')
ax1.tick_params(axis='y', labelcolor=color)

# Plot Sentiment on right Y-axis
ax2 = ax1.twinx()  
color = 'tab:blue'
ax2.set_ylabel('Average Text Sentiment', color=color, fontsize=12)
ax2.plot(time_agg['plot_date'], time_agg['text_sentiment'], color=color, linestyle='--', linewidth=2, label='Lyrics Sentiment')
ax2.tick_params(axis='y', labelcolor=color)

plt.title('Macro Evolution: COVID Trend vs. Lyrics Sentiment', fontsize=16, fontweight='bold')
fig.tight_layout()
plt.savefig(os.path.join(out_dir, '7B_TimeSeries_Trend_vs_Sentiment.png'), dpi=300)
plt.close()

print("✅ Stage 1 Visualizations saved.")

Generating Stage 1 Visualizations...
✅ Stage 1 Visualizations saved.


In [7]:
# ==========================================
# Phase 4: Stage 2 EDA - Bivariate Relationships & Correlation
# ==========================================
print("Generating Stage 2 Visualizations...")

# 1. Jointplot: Trend vs Valence (Is there a visual linear relationship?)
# Using a sample to prevent overplotting if dataset is too large
plot_sample = df_merged.sample(n=min(5000, len(df_merged)), random_state=42)

g = sns.jointplot(data=plot_sample, x='text_sentiment', y='covid_trend', kind='reg', 
                  scatter_kws={'alpha': 0.3, 's': 15}, line_kws={'color': 'red'})
g.fig.suptitle('Bivariate Relationship: Text Sentiment vs. COVID Trend', fontsize=14, fontweight='bold', y=1.03)
plt.savefig(os.path.join(out_dir, '7C_Jointplot_Sentiment_Trend.png'), dpi=300)
plt.close()

# 2. Advanced Correlation Heatmap (Checking for Multicollinearity before Regression)
regression_features = [
    'covid_trend', 'text_sentiment', 'lexical_richness', 
    'danceability', 'energy', 'valence', 'speechiness'
]

corr_matrix = df_merged[regression_features].corr(method='spearman')

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0, 
            square=True, linewidths=.5, cbar_kws={"shrink": .8})
plt.title('Correlation Matrix for Regression Variables', fontsize=16, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(out_dir, '7D_Regression_Vars_Correlation.png'), dpi=300)
plt.close()

print("✅ Stage 2 Visualizations saved.")

Generating Stage 2 Visualizations...
✅ Stage 2 Visualizations saved.


In [ ]:
# ==========================================
# Phase 5: Stage 3 EDA - Exploratory Statistical Modeling & Export
# ==========================================
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
import statsmodels.stats.api as sms
import os

print("Running Exploratory Statistical Models...")

# ---------------------------------------------------------
# Q1: T-Test (Comparing High vs Low COVID periods)
# ---------------------------------------------------------
print("\n--- Q1: Independent T-Test ---")
median_trend = df_merged['covid_trend'].median()
df_merged['trend_period'] = np.where(df_merged['covid_trend'] >= median_trend, 'High_COVID', 'Low_COVID')

high_covid_valence = df_merged[df_merged['trend_period'] == 'High_COVID']['valence'].dropna()
low_covid_valence = df_merged[df_merged['trend_period'] == 'Low_COVID']['valence'].dropna()

tstat, pvalue, dfree = sms.ttest_ind(high_covid_valence, low_covid_valence, usevar="unequal")

ttest_df = pd.DataFrame({
    'Metric': ["Valence (Musical Positivity)"],
    'Mean_High_COVID': [high_covid_valence.mean()],
    'Mean_Low_COVID': [low_covid_valence.mean()],
    'T_Statistic': [tstat],
    'P_Value': [pvalue],
    'Is_Significant (p<0.05)': [pvalue < 0.05]
})

print(f"T-statistic: {tstat:.4f} | P-value: {pvalue:.4e}")

# ---------------------------------------------------------
# Q2: Simple OLS Regression (Sentiment -> Trend)
# ---------------------------------------------------------
print("\n--- Q2: Simple OLS Regression (Sentiment predicting Trend) ---")
ols_model_1 = smf.ols("covid_trend ~ text_sentiment", data=df_merged).fit()

ols1_df = ols_model_1.summary2().tables[1].reset_index().rename(columns={'index': 'Variable'})
print(ols1_df)

# ---------------------------------------------------------
# Q3: Simple OLS Regression (Valence -> Trend)
# ---------------------------------------------------------
print("\n--- Q3: Simple OLS Regression (Valence predicting Trend) ---")
ols_model_2 = smf.ols("covid_trend ~ valence", data=df_merged).fit()

ols2_df = ols_model_2.summary2().tables[1].reset_index().rename(columns={'index': 'Variable'})
print(ols2_df)

# ==========================================
# Export Results to Excel
# ==========================================
print("\n💾 Exporting statistical results to Excel...")

excel_output_path = os.path.join(out_dir, '7E_EDA_Statistical_Results.xlsx')

# 使用 ExcelWriter 将不同的结果保存到不同的 Sheet 中
with pd.ExcelWriter(excel_output_path, engine='openpyxl') as writer:
    ttest_df.to_excel(writer, sheet_name='Q1_T_Test_Results', index=False)
    ols1_df.to_excel(writer, sheet_name='Q2_OLS_Sentiment', index=False)
    ols2_df.to_excel(writer, sheet_name='Q3_OLS_Valence', index=False)

print(f"✅ All statistical results successfully exported to: {excel_output_path}")

Running Exploratory Statistical Models...

--- Q1: Independent T-Test ---
Hypothesis: Average 'valence' (musical positivity) differs between High and Low COVID periods.
Mean Valence (High COVID): 0.4545
Mean Valence (Low COVID):  0.4501
T-statistic: 1.1434 | P-value: 2.5291e-01
Conclusion: Fail to reject the null hypothesis.

--- Q2: Simple OLS Regression (Sentiment predicting Trend) ---
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept         10.8929      0.169     64.448      0.000      10.562      11.224
text_sentiment     2.6765      0.873      3.067      0.002       0.966       4.387

--- Q3: Simple OLS Regression (Valence predicting Trend) ---
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     10.0646      0.338     29.769      0.000   

In [11]:
# ==========================================
# Phase 6: Final Preparation for Multiple Regression
# Checking Assumptions: Linearity & Multicollinearity
# ==========================================
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm

print("Preparing for Multiple Regression...")

# Define all the features you plan to put into the final model
melody_features = ['danceability', 'energy', 'loudness', 'speechiness', 'acousticness', 'valence']
all_predictors = ['text_sentiment'] + melody_features

# ---------------------------------------------------------
# Step 1: Feature vs Trend Linearity Check (Grid of Scatter Plots)
# ---------------------------------------------------------
print("1. Plotting Linearity checks for all predictors...")
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

# To prevent overplotting, take a random sample
plot_sample = df_merged.sample(n=min(3000, len(df_merged)), random_state=42)

for i, feature in enumerate(all_predictors):
    if feature in df_merged.columns:
        sns.regplot(data=plot_sample, x=feature, y='covid_trend', ax=axes[i],
                    scatter_kws={'alpha': 0.2, 's': 10}, line_kws={'color': 'red'})
        axes[i].set_title(f'Trend vs {feature.capitalize()}', fontweight='bold')

# Remove any empty subplots
for j in range(len(all_predictors), len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.savefig(os.path.join(out_dir, '7F_Linearity_Check_Grid.png'), dpi=300)
plt.close()

# ---------------------------------------------------------
# Step 2: Multicollinearity Check using VIF (Variance Inflation Factor)
# ---------------------------------------------------------
print("\n2. Calculating Variance Inflation Factor (VIF)...")
# Drop rows with NaN in these specific columns to run VIF
vif_data_clean = df_merged[all_predictors].dropna()

# Add a constant (intercept) to the variables, required for VIF
X_vif = sm.add_constant(vif_data_clean)

# Calculate VIF for each feature
vif_df = pd.DataFrame()
vif_df["Feature"] = X_vif.columns
vif_df["VIF_Score"] = [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]

# Filter out the constant and sort
vif_df = vif_df[vif_df['Feature'] != 'const'].sort_values('VIF_Score', ascending=False)
print("\n--- VIF Scores (Rule of thumb: VIF > 5 indicates problematic collinearity) ---")
print(vif_df.to_string(index=False))

# Optional: Export VIF results to Excel for your report
vif_excel_path = os.path.join(out_dir, '7G_VIF_Multicollinearity_Check.xlsx')
vif_df.to_excel(vif_excel_path, index=False)
print(f"\n✅ VIF Check saved to: {vif_excel_path}")

# ---------------------------------------------------------
# Conclusion / Guidance for your Final Model
# ---------------------------------------------------------
print("\n💡 EDA Recommendations for your Final Regression:")
high_vif_features = vif_df[vif_df['VIF_Score'] > 5]['Feature'].tolist()
if high_vif_features:
    print(f"⚠️ WARNING: The following features have high VIF (>5): {high_vif_features}.")
    print("   Consider dropping one of them (e.g., if Energy and Loudness are highly correlated, drop Loudness) to stabilize your regression coefficients.")
else:
    print("✅ Great! No severe multicollinearity detected among your chosen predictors.")

Preparing for Multiple Regression...
1. Plotting Linearity checks for all predictors...



2. Calculating Variance Inflation Factor (VIF)...

--- VIF Scores (Rule of thumb: VIF > 5 indicates problematic collinearity) ---
       Feature  VIF_Score
        energy   3.107460
      loudness   2.456535
  acousticness   1.983936
       valence   1.461877
  danceability   1.443395
   speechiness   1.138458
text_sentiment   1.040007

✅ VIF Check saved to: ..\result\figure\7_eda_regression\7G_VIF_Multicollinearity_Check.xlsx

💡 EDA Recommendations for your Final Regression:
✅ Great! No severe multicollinearity detected among your chosen predictors.
